## 1. Load the Dataset

The SECOM dataset is stored in two separate files:

- `secom.data` contains the anonymized measurement features.
- `secom_labels.data` contains the Pass/Fail labels and timestamps.

Before analysis, the two files must be loaded separately and checked for correct row alignment.

In [3]:
%pip install numpy pandas matplotlib seaborn scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 13.5 MB/s  0:00:00 8.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 10.4 MB/s  0:00:00.9 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 9.5 MB/s  0:00:00m 9.0 MB/s eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 7.8 MB/s  0:00:01a 0:00:010:00:01:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 9.0 MB/s  0:00:008.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 10.5 MB/s  0:00:001.3 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 8.7 MB/s  0:00:02m 8.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [seaborn]━ 14/16 [matplotlib]it-learn]
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
print("NumPy version:", np.__version__)
print("pandas version:", pd.__version__)

NumPy version: 2.5.3
pandas version: 3.0.6


In [3]:
Path.cwd()

PosixPath('<LOCAL_PATH>')

In [4]:
data_dir = Path("../data/raw")

print("Data directory exists:", data_dir.exists())
print("Files:", [file.name for file in data_dir.iterdir()])

Data directory exists: True
Files: ['secom.names', 'secom.data', '.ipynb_checkpoints', 'secom_labels.data']


### 1.1 Load and Verify the Feature Matrix

The UCI webpage reports 591 features, but the downloaded data file must be inspected directly.

Expected observations:

- 1,567 production samples
- Approximately 590 measurement features
- Missing values represented by `NaN`
- No original feature names

In [5]:
feature_path = data_dir / "secom.data"

X = pd.read_csv(
    feature_path,
    sep=r"\s+",
    header=None
)

In [6]:
n_samples, n_features = X.shape

print("Number of samples:", n_samples)
print("Number of features:", n_features)

Number of samples: 1567
Number of features: 590


In [8]:
X.columns = [f"feature_{i}" for i in range(X.shape[1])]

In [9]:
print("First five columns:", X.columns[:5].tolist())
print("Last five columns:", X.columns[-5:].tolist())

First five columns: ['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4']
Last five columns: ['feature_585', 'feature_586', 'feature_587', 'feature_588', 'feature_589']


In [10]:
X.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_580,feature_581,feature_582,feature_583,feature_584,feature_585,feature_586,feature_587,feature_588,feature_589
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,NaN,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,0.0060,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,0.0148,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,0.0044,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,NaN,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432


In [11]:
X.dtypes.value_counts()

float64    590
Name: count, dtype: int64

#### Initial Findings

- The feature matrix contains **1567 samples** and **590 measurement features**.
- The actual downloaded file contains 590 measurement columns, which differs from the 591 features reported on the UCI webpage.
- The original file does not provide feature names, so the columns were renamed from `feature_0` to `feature_589`.
- All measurement columns were loaded as float64 data.
- Missing values are represented as `NaN`.

### 1.2 Load and Verify the Labels

The label file contains the Pass/Fail outcome and test timestamp for each production sample.

This section checks:

1. Whether the label file contains 1,567 rows
2. Whether Pass and Fail are encoded as `-1` and `1`
3. How many Pass and Fail samples are present
4. Whether the timestamps can be parsed correctly
5. Whether the label rows align with the feature rows

In [13]:
label_path = data_dir / "secom_labels.data"

with open(label_path, "r") as file:
    for _ in range(5):
        print(file.readline().strip())

-1 "19/07/2008 11:55:00"
-1 "19/07/2008 12:32:00"
1 "19/07/2008 13:17:00"
-1 "19/07/2008 14:43:00"
-1 "19/07/2008 15:22:00"


In [14]:
labels = pd.read_csv(
    label_path,
    sep=r"\s+",
    header=None,
    names=["label", "timestamp_text"]
)

In [15]:
labels.head()

,label,timestamp_text
0,-1,19/07/2008 11:55:00
1,-1,19/07/2008 12:32:00
2,1,19/07/2008 13:17:00
3,-1,19/07/2008 14:43:00
4,-1,19/07/2008 15:22:00


In [16]:
print("Label data shape:", labels.shape)
print("Feature rows:", len(X))
print("Label rows:", len(labels))

Label data shape: (1567, 2)
Feature rows: 1567
Label rows: 1567


In [17]:
labels["label"].value_counts()

label
-1    1463
 1     104
Name: count, dtype: int64

In [18]:
print("Unique labels:", sorted(labels["label"].unique()))

Unique labels: [np.int64(-1), np.int64(1)]


In [19]:
label_counts = labels["label"].value_counts().sort_index()
label_percentages = labels["label"].value_counts(
    normalize=True
).sort_index().mul(100)

class_summary = pd.DataFrame({
    "count": label_counts,
    "percentage": label_percentages
})

class_summary

,count,percentage
label,,
-1,1463,93.363114
1,104,6.636886


In [20]:
class_summary.index = class_summary.index.map({
    -1: "Pass (-1)",
    1: "Fail (1)"
})

class_summary.round(2)

,count,percentage
label,,
Pass (-1),1463,93.36
Fail (1),104,6.64


In [21]:
labels["timestamp"] = pd.to_datetime(
    labels["timestamp_text"],
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

In [22]:
print("Timestamp data type:", labels["timestamp"].dtype)
print("Invalid timestamps:", labels["timestamp"].isna().sum())
print("Earliest timestamp:", labels["timestamp"].min())
print("Latest timestamp:", labels["timestamp"].max())

Timestamp data type: datetime64[us]
Invalid timestamps: 0
Earliest timestamp: 2008-07-19 11:55:00
Latest timestamp: 2008-10-17 06:07:00


In [23]:
assert len(X) == len(labels), (
    f"Row mismatch: X has {len(X)} rows, "
    f"but labels has {len(labels)} rows."
)

print("Feature and label row counts match.")

Feature and label row counts match.


### 1.3 Combine Features, Labels, and Timestamps

The feature file and label file contain the same number of rows. Since no sample ID is provided, this analysis matches them by their original row order.

Before combining them, I will verify their row counts and indexes. After combining them, I will check several rows and the final table shape.

In [24]:
print("Feature shape:", X.shape)
print("Label shape:", labels.shape)

assert len(X) == len(labels)
assert X.index.equals(labels.index)

print("Row counts and indexes match.")

Feature shape: (1567, 590)
Label shape: (1567, 3)
Row counts and indexes match.


In [28]:
df = pd.concat(
    [X, labels[["label", "timestamp"]]],
    axis=1
)

print("Combined shape:", df.shape)
df[["feature_0", "feature_1", "label", "timestamp"]].head()

Combined shape: (1567, 592)


,feature_0,feature_1,label,timestamp
0,3030.93,2564.00,-1,2008-07-19 11:55:00
1,3095.78,2465.14,-1,2008-07-19 12:32:00
2,2932.61,2559.94,1,2008-07-19 13:17:00
3,2988.72,2479.90,-1,2008-07-19 14:43:00
4,3032.24,2502.87,-1,2008-07-19 15:22:00


In [29]:
print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Missing labels:", df["label"].isna().sum())
print("Missing timestamps:", df["timestamp"].isna().sum())
print("Label values:", sorted(df["label"].unique()))

Rows: 1567
Columns: 592
Missing labels: 0
Missing timestamps: 0
Label values: [np.int64(-1), np.int64(1)]


In [30]:
n_total = len(df)
n_pass = (df["label"] == -1).sum()
n_fail = (df["label"] == 1).sum()

always_pass_accuracy = n_pass / n_total
always_pass_fail_recall = 0 / n_fail

print("Total samples:", n_total)
print("Actual Pass samples:", n_pass)
print("Actual Fail samples:", n_fail)
print("Always-Pass accuracy:", round(always_pass_accuracy, 4))
print("Always-Pass Fail recall:", always_pass_fail_recall)

Total samples: 1567
Actual Pass samples: 1463
Actual Fail samples: 104
Always-Pass accuracy: 0.9336
Always-Pass Fail recall: 0.0


#### Baseline Observation

An Always-Pass rule achieves an accuracy of approximately 93.36%, because there are 1463 pass samples and they are precited correctly.

However, its Fail recall is 0%, meaning that we found zero fail sample but there are 104 truly fail sample, and fail recall equals to 0 deived by 104 always equal to 0.

herefore, when evaluating future models, I should examine Fail recall and precision alongside accuracy. Fail recall measures how many actual failures are detected, while precision measures how many Fail alerts are correct.

### 1.4 Audit Missing Values

Missing values may affect which features can be used and how the data is prepared for modeling.

Questions:

1. What percentage of all measurement entries is missing?
2. How many features contain at least one missing value?
3. Which features have the highest missing rates?
4. Are any features entirely missing?

In [31]:
total_entries = X.size
missing_entries = X.isna().sum().sum()
overall_missing_rate = missing_entries / total_entries

print("Total measurement entries:", total_entries)
print("Missing measurement entries:", missing_entries)
print("Overall missing rate:", round(overall_missing_rate * 100, 2), "%")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Total measurement entries: 924530
Missing measurement entries: 41951
Overall missing rate: 4.54 %


In [32]:
missing_rate_by_feature = X.isna().mean().sort_values(ascending=False)

print("Features with at least one missing value:",
      (missing_rate_by_feature > 0).sum())

print("Completely missing features:",
      (missing_rate_by_feature == 1).sum())

missing_rate_by_feature.head(10)

Features with at least one missing value: 538
Completely missing features: 0


feature_157    0.911934
feature_292    0.911934
feature_293    0.911934
feature_158    0.911934
feature_492    0.855775
feature_358    0.855775
feature_85     0.855775
feature_220    0.855775
feature_246    0.649649
feature_109    0.649649
dtype: float64

In [33]:
(missing_rate_by_feature.head(10) * 100).round(2)

feature_157    91.19
feature_292    91.19
feature_293    91.19
feature_158    91.19
feature_492    85.58
feature_358    85.58
feature_85     85.58
feature_220    85.58
feature_246    64.96
feature_109    64.96
dtype: float64

#### Should Features With High Missing Rates Be Removed?

A high missing rate alone does not prove that a feature is useless. Its observed values may help predict Fail, and whether the measurement is missing may itself contain information. However, conclusions based on very few observed samples may be unstable.

For a feature with about 91% missing values, I should first check how many Pass and Fail samples have an observed value, and how many have a missing value. I should compare the Fail rates in these two groups, rather than looking only at the average of the observed measurements.

This comparison is exploratory: a difference in Fail rates would show an association, not explain why the measurement is missing or prove a physical cause. I will record highly missing features for further evaluation before deciding whether to keep or remove them.

### 1.5 Missing Values by Sample

The previous section measured missingness by feature. This section checks how many measurements are missing for each production sample.

In [35]:
missing_count_by_sample = X.isna().sum(axis=1)
missing_rate_by_sample = X.isna().mean(axis=1)

print("Samples with at least one missing value:",
      (missing_count_by_sample > 0).sum())

print("Samples with no missing values:",
      (missing_count_by_sample == 0).sum())

print("Median missing measurements per sample:",
      missing_count_by_sample.median())

print("Maximum missing measurements in one sample:",
      missing_count_by_sample.max())

Samples with at least one missing value: 1567
Samples with no missing values: 0
Median missing measurements per sample: 24.0
Maximum missing measurements in one sample: 152


In [36]:
(missing_rate_by_sample.sort_values(ascending=False).head(5) * 100).round(2)

1566    25.76
1564    25.08
1561    23.73
1152    16.95
511     16.95
dtype: float64

### 1.6 Constant Feature Audit

A constant feature has the same value in every observed sample, so its measured value cannot distinguish one sample from another.

This section counts constant features without changing the original data. Missing values are ignored when counting distinct observed values.

In [37]:
unique_values_per_feature = X.nunique(dropna=True)
constant_features = unique_values_per_feature[
    unique_values_per_feature == 1
]

print("Number of constant features:", len(constant_features))
print("Constant feature names:", constant_features.index.tolist())

Number of constant features: 116
Constant feature names: ['feature_5', 'feature_13', 'feature_42', 'feature_49', 'feature_52', 'feature_69', 'feature_97', 'feature_141', 'feature_149', 'feature_178', 'feature_179', 'feature_186', 'feature_189', 'feature_190', 'feature_191', 'feature_192', 'feature_193', 'feature_194', 'feature_226', 'feature_229', 'feature_230', 'feature_231', 'feature_232', 'feature_233', 'feature_234', 'feature_235', 'feature_236', 'feature_237', 'feature_240', 'feature_241', 'feature_242', 'feature_243', 'feature_256', 'feature_257', 'feature_258', 'feature_259', 'feature_260', 'feature_261', 'feature_262', 'feature_263', 'feature_264', 'feature_265', 'feature_266', 'feature_276', 'feature_284', 'feature_313', 'feature_314', 'feature_315', 'feature_322', 'feature_325', 'feature_326', 'feature_327', 'feature_328', 'feature_329', 'feature_330', 'feature_364', 'feature_369', 'feature_370', 'feature_371', 'feature_372', 'feature_373', 'feature_374', 'feature_375', 'feat

In [38]:
unique_values_per_feature.value_counts().sort_index().head(10)

1     116
2       6
9       1
14      1
20      2
21      1
22      3
23      1
28      1
29      2
Name: count, dtype: int64

## Data Audit Summary

- The downloaded feature matrix contains **1,567 samples and 590 anonymized measurement features**. The UCI webpage reports 591 features; this analysis uses the dimensions verified from the downloaded file.
- The target contains **1,463 Pass (-1)** and **104 Fail (1)** samples. Fail accounts for **6.64%** of all samples.
- An Always-Pass rule achieves **93.36% accuracy** but **0% Fail recall**. Accuracy alone is therefore insufficient for evaluating failure detection.
- The measurement matrix contains **41,951 missing entries out of 924,530 (4.54%)**. Missingness is uneven across features: **538 features** have at least one missing value, and the highest feature missing rate is **91.19%**.
- Every sample has at least one missing measurement. The median is **24 missing measurements per sample**, and the maximum is **152**.
- **116 features** have only one distinct observed value when missing values are ignored. Their observed measurements cannot distinguish samples, although missingness may still need consideration.
- All labels and timestamps were parsed without missing values. The feature and label files have matching row counts and indexes; their pairing relies on the original file order because no independent sample ID is provided.

### Decisions for the next stage

I will preserve the raw data during exploration. Decisions about imputation and feature removal will be learned from training data during modeling.

The audit results describe the full dataset. During cross-validation, imputation, constant-feature removal, scaling, and feature selection will be fitted separately within each training fold.